[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sterolose/sirius-statistics-2026/blob/main/notebooks/7_anova.ipynb)

## Семинар 7. Дисперсионный анализ

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
import pyartool as art

Вернемся к датасету "Palmer Penguins". Подготовим его как раньше.

In [ ]:
df = sns.load_dataset("penguins")
df.insert(2, "body_mass_kg", df.pop("body_mass_g") / 1000)
df = df.dropna()
df.head(2)

Во всех заданиях используем уровень значимости $\alpha = 0.05$, в парных сравнениях — двусторонние альтернативы.

### Задание 1

Применим однофакторный дисперсионный анализ для сравнения средней массы между полами.

In [ ]:
male = df.loc[df['sex'] == 'Male', 'body_mass_kg']
female = df.loc[df['sex'] == 'Female', 'body_mass_kg']

result = stats.f_oneway(male, female)
f"statistic = {result.statistic:.2f}, p-value = {result.pvalue:.3g}"

Сравните результат с t-тестом Стьюдента.

In [ ]:
# Your code here

### Задание 2

Теперь оценим различия средней массы тела между видами. Однако напомню, что классическая ANOVA предполагает равенство дисперсий между группами. Проверим это предположение с помощью тестов Бартлетта и Левене.

In [ ]:
groups = df.groupby('species')['body_mass_kg'].apply(list)

pd.DataFrame({
    "test": ["Bartlett", "Levene"],
    "p-value": [
        stats.bartlett(*groups).pvalue,
        stats.levene(*groups).pvalue,
    ],
})

Примените однофакторный ANOVA к тем же группам, указав необходимое значение параметра `equal_var`.

In [ ]:
# Your code here

Также ANOVA не отвечает на вопрос, средние каких именно групп различаются. Для каждой пары групп выполните тест Уэлча (можете использовать `combinations` для перебора всех пар). Примените поправку Бонферрони и выведите результат как `pd.DataFrame`.

In [ ]:
# Your code here

При равных дисперсиях для всех парных сравнений удобно применять тест Тьюки. Для аналога с неравными дисперсиями — теста Геймса — Хауэлла — укажем `use_var="unequal"`.

In [ ]:
pairwise_tukeyhsd(df["body_mass_kg"], df["species"], use_var="unequal").summary()

Результат может быть визуализирован.

In [ ]:
tukey = pairwise_tukeyhsd(df["body_mass_kg"], df["species"])
tukey.plot_simultaneous(figsize=(7, 3))
plt.show()

### Задание 3

Давайте применим двухфакторный ANOVA для вида и пола птицы. Используем `ols` и `anova_lm` из `statsmodels` с R-подобным синтаксисом. Будем считать дисперсии в группах «вид × пол» одинаковыми; проверку пропустим.

In [ ]:
model = ols("body_mass_kg ~ species + sex", df).fit()
anova_lm(model)

До этого мы не рассматривали взаимодействие факторов. То есть предполагали, что разница средней массы между полами одинакова у разных видов. Чтобы учесть различия полового диморфизма между видами, можно использовать запись `species + sex + species:sex` либо сокращенный вариант `species * sex`.

In [ ]:
model = ols("body_mass_kg ~ species * sex", df).fit()
anova_lm(model)

Напомню, что ANOVA предполагает нормальность ошибок модели, а не исходной переменной в целом. Проверим это предположение по остаткам: постройте гистограмму, Q–Q plot и выполните тест Шапиро — Уилка. Получить остатки модели можно как `model.resid`.

In [ ]:
# Your code here

Выполните тест Тьюки и визуализируйте результат. Тест не умеет работать с несколькими категориальными переменными, объединим их в одну.

In [ ]:
df['group'] = df['species'] + '-' + df['sex']

# Your code here

Различия можно визуализировать и с более красочным и понятным графиком.

In [ ]:
sns.pointplot(
    df, x="species", y="body_mass_kg", hue="sex",
    errorbar=("ci", 95), capsize=0.05,
    linewidth=1.5, markersize=4,
)
plt.show()

Однако такой график не имеет поправки на множественные сравнения и не заменяет тесты.

### Задание 4

Теперь давайте оценим различие средней длины клюва в группах «вид × пол». Выполните тест Левена для оценки равенства дисперсий в группах. Постройте двухфакторную ANOVA. Проверьте нормальность остатков.

In [ ]:
model = ols("bill_length_mm ~ species * sex", df).fit()
anova_lm(model)

In [ ]:
# Your code here

При ненормальности остатков следует применять тест Краскела — Уоллиса. Недостатком теста является его однофакторность, и следовательно не может разделять основные эффекты и взаимодействия. Представим два фактора как один с шестью уровнями.

In [ ]:
df['group'] = df['species'] + '-' + df['sex']
groups = df.groupby('group')['bill_length_mm'].apply(list)

result = stats.kruskal(*groups)
f"statistic = {result.statistic:.2f}, p-value = {result.pvalue:.3g}"

Непараметрическим аналогом теста Тьюки является тест Данна.

In [ ]:
sp.posthoc_dunn(df, val_col="bill_length_mm", group_col="group", p_adjust="holm")

Давайте сделаем вывод немного красочнее.

In [ ]:
matrix = sp.posthoc_dunn(df, val_col="bill_length_mm", group_col="group", p_adjust="holm")

sns.heatmap(matrix, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=0.05)
plt.show()

Для непараметрического анализа влияния нескольких факторов и их взаимодействий можно использовать Aligned Rank Transform (ART). У этого метода также есть предпосылки, которые необходимо проверять перед интерпретацией результатов.

In [ ]:
model = art.art("bill_length_mm ~ species * sex", df)
art.anova_art(model)

Для парных сравнений метод имеет собственный постхок ART-C.

In [ ]:
art.art_con(model, "species:sex", adjust="tukey")

### Задание 5

Самостоятельно оцените различия `bill_depth_mm` в группах «вид × пол» с помощью ANOVA. Проверьте предпосылки модели. При соблюдении предпослылок выполните тест Тьюки. При их несоблюдении — тесты Краскела — Уоллиса и Данна. Визуализируйте результат желаемым образом.

In [ ]:
# Your code here